# The public API

`tri.api` is the one public module; every other name may change between releases. The library takes its inputs as arguments. It never reads the environment, never opens a network connection and reads and writes local run files only. Every call returns JSON values.

| call | returns |
|---|---|
| `load_run(run_dir, catalog=None)` | a `Tri` from `<run>/model` and `<run>/catalog.json` |
| `with_reviews(tri, reviews)` | a new `Tri` with this review log; `tri` is unchanged |
| `with_memory(tri, reviews, locators)` | a new `Tri` with these reviews and locators |
| `review_build(tri, build, n=3, archetype=None, locked=(), answers=None)` | up to `n` answers, best archetype first: the smallest edits to a full build |
| `complete_build(tri, build, archetype=None, answers=None)` | the build filled to its talent cap and mantra slots |
| `dry_insert(tri, review, probes=())` | what one review would change, without changing `tri` |
| `agreement(tri)` | each active review answered on its own build |
| `evaluate(tri, workers=2, B=1000)` | leave-one-build-out metrics with standard errors (notebook 04) |

An answer holds `talents`, `mantras`, `changes` (each `{item, action, reason, related, worth, source, review_id, pass_, why}`), `iffy`, `capacity` (talent cap, mantra slots), `concept`, `name` (archetype), `plausibility`, `violations`, `sidegrades`, `unrated` and `focus`.

**Needs a run.** Set `RUN` to a fitted run directory and `SNAPSHOT` to a table snapshot written by `python -m bot.evaluate snapshot snap.json`.

In [ ]:
RUN = "runs/live"          # a fitted run: model/ and catalog.json
SNAPSHOT = "snap.json"     # python -m bot.evaluate snapshot snap.json

## Loading

A run stores the fitted model only. Reviews are handed in afterwards, and `with_reviews` returns a fork that shares the model and owns its review state. A new review therefore acts at once, with no refit.

In [ ]:
import copy, json, time
from ipywidgets import interact, IntSlider
from tri import api

t = time.time()
base = api.load_run(RUN)
rows = json.load(open(SNAPSHOT))["reviews"]
reviews = []
for row in rows:
    body = row["operations"]
    body = json.loads(body) if isinstance(body, str) else body
    try:
        r = api.review_from_dict(body)
        api.validate_review(copy.deepcopy(r), base.catalog)   # raises on items this run does not know
        reviews.append(r)
    except Exception:          # audits, rows without a build snapshot and invalid reviews are skipped
        pass
tri = api.with_reviews(base, reviews)
print(f"{len(reviews)} reviews, loaded in {time.time() - t:.1f} s")

## Reviewing a build

`review_build` answers a full build with the smallest set of edits. Pick a reviewed build; the cell shows the reviewer's operations beside tri's. With the full log the review binds on its own build, so its operations come back (review agreement); tri may also propose edits of its own. Notebook 04 measures the harder case, where the build's own reviews are held out.

In [ ]:
def ops(r):
    return {(o.item_id, o.action.value) for o in r.operations if o.action.value in ("add", "remove")}

def show_review(i=0):
    r = reviews[i]
    a = api.review_build(tri, r.original, n=1)[0]
    mine = {(c["item"], c["action"]) for c in a["changes"] if c["action"] in ("add", "remove")}
    print(f"archetype {a['name']}, capacity {a['capacity']}")
    for item, act in sorted(ops(r) | mine, key=lambda x: (x[1], x[0])):
        tag = "both" if (item, act) in ops(r) and (item, act) in mine else ("reviewer" if (item, act) in ops(r) else "tri")
        print(f"  {act:6} {item:40} {tag}")

interact(show_review, i=IntSlider(0, 0, len(reviews) - 1));

## Completing a partial build

`complete_build` fills every talent and mantra slot: prerequisites go in first, mantras before talents (talents first on Silentheart), and wildcard slots take any mantra type. Drop some talents from a reviewed build and see which come back. Mantras are filled first, and each mantra lowers the talent cap by 2.

In [ ]:
import random

def show_complete(i=0, drop=10, seed=0):
    b = reviews[i].original
    rnd = random.Random(seed)
    kept = [t for t in b.talents if rnd.random() > drop / max(1, len(b.talents))]
    part = api.build_from_dict({**api.to_json(b), "talents": kept})
    a = api.complete_build(tri, part)
    back = set(a["talents"]) & (set(b.talents) - set(kept))
    cap, slots = a["capacity"]
    print(f"dropped {len(b.talents) - len(kept)} talents; {len(back)} of them came back")
    print(f"talents {len(a['talents'])}/{cap} (cap 76 - 2m), mantras {len(a['mantras'])}/{slots}")

interact(show_complete, i=IntSlider(0, 0, len(reviews) - 1), drop=IntSlider(10, 0, 40), seed=IntSlider(0, 0, 20));

## What one review would change

`dry_insert` answers the review's build (and any `probes`) before and after the review, lists the older claims it supersedes and the claims it reproduces on its own build. `tri` is not changed, so it is safe beside running answers.

In [ ]:
r = reviews[-1]
d = api.dry_insert(base, r, probes=[x.original for x in reviews[:3]])
print({k: (len(v) if isinstance(v, (list, dict)) else v) for k, v in d.items()})

## Inputs and shared state

| input | type | where it comes from |
|---|---|---|
| catalog | `Catalog` | `<run>/catalog.json`, or `Catalog.from_json` |
| builds | `Build` | `build_from_dict`, or the builder parsers in `dwt` |
| reviews | `Review` | `review_from_dict`; validated by `validate_review` |
| locators | `Locator` | archetype anchors kept by the caller |
| authored rules | rule language text | `Tri.add_rule`, `Tri.set_authored` |
| game data | files | `load_gamedata`, `load_deepwokenco`, `merge_gamedata`; downloading them is the caller's job |

`Tri.set_reviews`, `set_memory`, `add_reviews` and `apply_patch` change the object they are called on; `Tri.fork()`, `with_reviews`, `with_memory` and `dry_insert` work on a copy. An async caller builds the next state on a fork and swaps it in while answers run on the current one.

Fitting a run:

```
python -m tri.cli.fit --builds builds.json --out runs --name live --gamedata PATH
python -m tri.cli.examine runs/live
```

`tests/test_boundary.py` fails if any `tri` module imports `bot` or a network module or reads the environment, or if the bot imports from `tri` other than `tri.api`.